In [2]:
# filepath: ./usi_usopen_2025.ipynb (paste cells from this monolithic block)
"""
USI (Upset Susceptibility Index) for US Open 2025.

Inputs (auto-discovered, first existing):
  - Dataset snapshot (pre-USO preferred):
      ./data/match_dataset_post_cincinnati_2025.csv
      ./data/match_dataset_post_wimbledon_2025_canada.csv
      ./data/match_dataset_post_wimbledon_2025.csv
      ./atp_matches_2021_2024.csv
      /mnt/data/atp_matches_2021_2024.csv
  - Combined USO predictions (preferred) OR per-round fallbacks:
      ./reports/usopen2025_all_rounds_predictions_combined.csv
      ./reports/usopen2025_R128_predictions*.csv … F

Outputs:
  - ./reports/usi/player_usi.csv
  - ./reports/usopen2025_all_rounds_predictions_with_usi.csv
"""

from __future__ import annotations

import os, re, unicodedata
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd

# ---------------- Config ----------------
PREF_DATASETS = [
    "./data/match_dataset_post_cincinnati_2025.csv",
    "./data/match_dataset_post_wimbledon_2025_canada.csv",
    "./data/match_dataset_post_wimbledon_2025.csv",
    "./atp_matches_2021_2024.csv",
    "/mnt/data/atp_matches_2021_2024.csv",
]
USO_COMBINED = "./reports/usopen2025_all_rounds_predictions_combined.csv"
USO_ROUND_FILES = [
    "./reports/usopen2025_R128_predictions_complete.csv",
    "./reports/usopen2025_R64_predictions_complete.csv",
    "./reports/usopen2025_R32_predictions_complete.csv",
    "./reports/usopen2025_R16_predictions_complete.csv",
    "./reports/usopen2025_QF_predictions_complete.csv",
    "./reports/usopen2025_SF_predictions_complete.csv",
    "./reports/usopen2025_F_predictions_complete.csv",
    "./reports/usopen2025_R128_predictions.csv",
    "./reports/usopen2025_R64_predictions.csv",
    "./reports/usopen2025_R32_predictions.csv",
    "./reports/usopen2025_R16_predictions.csv",
    "./reports/usopen2025_QF_predictions.csv",
    "./reports/usopen2025_SF_predictions.csv",
    "./reports/usopen2025_F_predictions.csv",
]

USO_TOURNEY_NAME = "US Open 2025"
USO_SURFACE = "Hard"

# USI hyper-params (tweak later)
LOOKBACK_DAYS = 1000
FAV_THRESH = 0.60
PRIOR_STRENGTH = 20.0          # EB strength toward field upset rate among favorites
K_OVERALL = 24.0               # Elo updates used to construct expectation
K_SURFACE = 18.0
VOL_WINDOW = 20                # volatility residual window
TREND_DAYS = 180               # trend slope horizon (days)
FATIGUE_DAYS = 14
VOL_SCALE = 0.50               # residual std ~ [0,0.5]
TREND_FULLSCALE_ELO = 50.0     # 50 Elo drop over TREND_DAYS => score ~ 1.0
FATIGUE_FULLSCALE_MATCHES = 6  # 6 matches in 14 days => score ~ 1.0

WEIGHTS = {                    # why: emphasize core brittleness, then volatility/trend
    "z": 0.45,
    "vol": 0.20,
    "trend": 0.15,
    "fatigue": 0.10,
    "surf_mismatch": 0.10,
}

OUT_DIR = "./reports/usi"
OUT_USI_PLAYERS = os.path.join(OUT_DIR, "player_usi.csv")
OUT_USO_WITH_USI = "./reports/usopen2025_all_rounds_predictions_with_usi.csv"

# ---------------- Utilities ----------------
def _ensure_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

def _first_existing(paths: List[str]) -> Optional[str]:
    for p in paths:
        if os.path.exists(p):
            return p
    return None

def _safe_date(x) -> pd.Timestamp:
    s = str(x)
    try:
        if len(s) == 8 and s.isdigit():
            return pd.to_datetime(s, format="%Y%m%d")
        return pd.to_datetime(s)
    except Exception:
        return pd.to_datetime(s, errors="coerce")

def normalize_surface(s: str) -> str:
    if pd.isna(s): return "Other"
    s = str(s).strip().title()
    if "Clay" in s: return "Clay"
    if "Grass" in s: return "Grass"
    if "Hard" in s or "Acrylic" in s or "Carpet" in s: return "Hard"
    return "Other"

def canon(s: str) -> str:
    s = unicodedata.normalize("NFKD", s or "").encode("ascii","ignore").decode("ascii")
    return re.sub(r"[^a-z0-9]+"," ", s.lower()).strip()

ALIAS = {
    "Felix Auger-Aliassime": "Felix Auger Aliassime",
    "Alex de Minaur": "Alex De Minaur",
    "Botic Van De Zandschulp": "Botic van de Zandschulp",
    "Giovanni Mpetshi-Perricard": "Giovanni Mpetshi Perricard",
}
def alias(s: str) -> str:
    return ALIAS.get(s, s)

# ---------------- Player log (pre/post variables) ----------------
@dataclass
class EloParams:
    base: float = 1500.0
    k_overall: float = K_OVERALL
    k_surface: float = K_SURFACE

def build_player_log(matches: pd.DataFrame, params: EloParams) -> pd.DataFrame:
    need = ["tourney_date","tourney_name","surface","tourney_level","match_num",
            "winner_name","loser_name","round","best_of"]
    for c in need:
        if c not in matches.columns:
            raise ValueError(f"Missing required column: {c}")

    m = matches.copy()
    m["tourney_date"] = m["tourney_date"].apply(_safe_date)
    m["surface"] = m["surface"].map(normalize_surface)
    m["winner_name"] = m["winner_name"].astype(str).map(alias)
    m["loser_name"]  = m["loser_name"].astype(str).map(alias)
    m = m.sort_values(["tourney_date","tourney_name","match_num"], kind="mergesort").reset_index(drop=True)

    BASE, K, KS = params.base, params.k_overall, params.k_surface
    elo_overall: Dict[str,float] = {}
    elo_surface: Dict[Tuple[str,str],float] = {}

    rows: List[Dict] = []
    for _, r in m.iterrows():
        d = r["tourney_date"]; s = r["surface"]
        w = r["winner_name"]; l = r["loser_name"]
        ew_w = elo_overall.get(w, BASE); ew_l = elo_overall.get(l, BASE)
        es_w = elo_surface.get((w, s), BASE); es_l = elo_surface.get((l, s), BASE)

        exp_w   = 1.0/(1.0+10**((ew_l-ew_w)/400))
        exp_w_s = 1.0/(1.0+10**((es_l-es_w)/400))

        new_ew_w = ew_w + K  * (1-exp_w)
        new_ew_l = ew_l - K  * (1-exp_w)
        new_es_w = es_w + KS * (1-exp_w_s)
        new_es_l = es_l - KS * (1-exp_w_s)

        rows.append({"player": w, "opp": l, "date": d, "surface": s, "is_win": 1,
                     "pre_elo": ew_w, "pre_selo": es_w, "opp_pre_elo": ew_l, "opp_pre_selo": es_l,
                     "post_elo": new_ew_w, "post_selo": new_es_w,
                     "tourney_name": r["tourney_name"], "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"]})
        rows.append({"player": l, "opp": w, "date": d, "surface": s, "is_win": 0,
                     "pre_elo": ew_l, "pre_selo": es_l, "opp_pre_elo": ew_w, "opp_pre_selo": es_w,
                     "post_elo": new_ew_l, "post_selo": new_es_l,
                     "tourney_name": r["tourney_name"], "tourney_level": r["tourney_level"], "round": r["round"], "best_of": r["best_of"]})

        elo_overall[w] = new_ew_w; elo_overall[l] = new_ew_l
        elo_surface[(w, s)] = new_es_w; elo_surface[(l, s)] = new_es_l

    log = pd.DataFrame(rows).sort_values(["player","date"], kind="mergesort").reset_index(drop=True)
    log["player_key"] = log["player"].map(canon); log["opp_key"] = log["opp"].map(canon)

    # rolling (post) residuals and counts
    grp = log.groupby("player_key")
    # residuals per match vs expected (surface expected if available)
    pwin_surf = 1.0/(1.0 + 10**((log["opp_pre_selo"] - log["pre_selo"]) / 400.0))
    pwin_over = 1.0/(1.0 + 10**((log["opp_pre_elo"] - log["pre_elo"]) / 400.0))
    log["pwin"] = np.where(log["surface"].notna(), pwin_surf, pwin_over)
    log["residual"] = log["is_win"] - log["pwin"]

    # simple rolling counts for fatigue later (inclusive post)
    from collections import deque
    def rolling_counts(dates: List[pd.Timestamp], window: int) -> List[int]:
        dq = deque(); out=[]
        for dt in dates:
            while dq and (dt - dq[0]).days > window: dq.popleft()
            out.append(len(dq)+1); dq.append(dt)
        return out
    log["matches_14d_post"] = grp["date"].transform(lambda s: rolling_counts(list(s), 14))
    log["matches_28d_post"] = grp["date"].transform(lambda s: rolling_counts(list(s), 28))

    # smoothed WR overall & per surface (post)
    SM_A = 5.0; SM_B = 5.0
    wins_post = grp["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    cnt_post  = grp["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    log["wr_overall_post_sm"] = ((SM_A + wins_post) / (SM_A + SM_B + cnt_post)).clip(0,1)

    log_sorted = log.sort_values(["player_key","surface","date"])
    wins_surf = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().sum()).fillna(0.0)
    cnt_surf  = log_sorted.groupby(["player_key","surface"])["is_win"].transform(lambda s: s.expanding().count()).fillna(0.0)
    wr_surf_sm = ((SM_A + wins_surf) / (SM_A + SM_B + cnt_surf)).clip(0,1)
    tmp = log_sorted[["player_key","date","surface"]].copy()
    tmp["wr_surf_post_sm"] = wr_surf_sm.values
    surf_post = tmp.pivot_table(index=["player_key","date"], columns="surface", values="wr_surf_post_sm", aggfunc="last")
    for srf in ["Clay","Grass","Hard"]:
        if srf not in surf_post.columns: surf_post[srf] = np.nan
    surf_post = surf_post[["Clay","Grass","Hard"]].sort_index().reset_index()
    surf_post[["Clay","Grass","Hard"]] = surf_post.groupby("player_key")[["Clay","Grass","Hard"]].ffill().fillna(0.5)

    log = log.merge(surf_post.rename(columns={"Clay":"wr_Clay_post_sm","Grass":"wr_Grass_post_sm","Hard":"wr_Hard_post_sm"}),
                    on=["player_key","date"], how="left")
    return log

# ---------------- USI components ----------------
def favorite_mask(df: pd.DataFrame, asof_end: pd.Timestamp, lookback_days: int) -> pd.Series:
    start = asof_end - pd.Timedelta(days=lookback_days)
    return (df["date"] >= start) & (df["date"] <= asof_end) & (df["pwin"] >= FAV_THRESH)

def compute_core_brittleness(df_player: pd.DataFrame, asof_end: pd.Timestamp, lookback_days: int,
                             field_q0: float) -> Dict[str, float]:
    m = df_player.loc[favorite_mask(df_player, asof_end, lookback_days)].copy()
    N = int(len(m))
    if N == 0:
        return {"N_fav":0,"O_upsets":0,"E_exp":0.0,"V_var":0.0,"z":0.0,"eb_rate":field_q0}
    # expected loss prob = (1 - pwin) for favorites
    E = float((1.0 - m["pwin"]).sum())
    V = float((m["pwin"] * (1.0 - m["pwin"])).sum())
    O = int((m["is_win"] == 0).sum())
    z = 0.0
    if V > 1e-8:
        z = (O - E) / np.sqrt(V)  # why: proper standardized residual
    # EB toward field upset rate
    obs_rate = O / N
    eb_rate = (obs_rate * N + field_q0 * PRIOR_STRENGTH) / (N + PRIOR_STRENGTH)
    return {"N_fav":N,"O_upsets":O,"E_exp":E,"V_var":V,"z":float(z),"eb_rate":float(eb_rate)}

def sigmoid(x: float) -> float:
    return 1.0/(1.0 + np.exp(-x))

def component_volatility(df_player: pd.DataFrame, asof_end: pd.Timestamp) -> float:
    sub = df_player[df_player["date"] <= asof_end].tail(VOL_WINDOW)
    if sub.empty:
        return 0.0
    std = float(sub["residual"].std(ddof=0)) if len(sub) > 1 else 0.0
    return float(np.clip(std / VOL_SCALE, 0.0, 1.0))

def component_trend(df_player: pd.DataFrame, asof_end: pd.Timestamp) -> float:
    start = asof_end - pd.Timedelta(days=TREND_DAYS)
    sub = df_player[(df_player["date"] >= start) & (df_player["date"] <= asof_end)][["date","pre_elo"]].copy()
    if len(sub) < 3:
        return 0.0
    t = (sub["date"] - sub["date"].min()).dt.days.astype(float).values
    y = sub["pre_elo"].astype(float).values
    # OLS slope
    t_mean = t.mean(); y_mean = y.mean()
    denom = np.sum((t - t_mean)**2)
    if denom <= 1e-9:
        return 0.0
    slope_per_day = float(np.sum((t - t_mean)*(y - y_mean)) / denom)
    # negative slope → higher score
    scaled = (-slope_per_day * TREND_DAYS) / TREND_FULLSCALE_ELO
    return float(np.clip(scaled, 0.0, 1.0))

def component_fatigue(df_player: pd.DataFrame, asof_end: pd.Timestamp) -> float:
    start = asof_end - pd.Timedelta(days=FATIGUE_DAYS)
    n = int(((df_player["date"] >= start) & (df_player["date"] <= asof_end)).sum())
    return float(np.clip(n / FATIGUE_FULLSCALE_MATCHES, 0.0, 1.0))

def component_surface_mismatch(snapshot_row: pd.Series) -> float:
    overall = float(snapshot_row.get("wr_overall_post_sm", np.nan))
    hard = float(snapshot_row.get("wr_Hard_post_sm", np.nan))
    if np.isnan(overall) or np.isnan(hard):
        return 0.0
    mis = max(0.0, overall - hard)
    return float(np.clip(mis, 0.0, 1.0))

# ---------------- USO discovery / assembly ----------------
def load_uso_predictions() -> pd.DataFrame:
    if os.path.exists(USO_COMBINED):
        return pd.read_csv(USO_COMBINED)
    dfs=[]
    for p in USO_ROUND_FILES:
        if os.path.exists(p):
            dfs.append(pd.read_csv(p))
    if not dfs:
        raise FileNotFoundError("US Open prediction CSVs not found. Provide combined or per-round files.")
    big = pd.concat(dfs, ignore_index=True)
    # enforce basic order if present
    if "round" in big.columns and "match_no" in big.columns:
        order = {"R128":1,"R64":2,"R32":3,"R16":4,"QF":5,"SF":6,"F":7}
        big["__ro__"] = big["round"].map(order).fillna(99).astype(int)
        big["__mn__"] = pd.to_numeric(big["match_no"], errors="coerce")
        big = big.sort_values(["__ro__","__mn__"], kind="mergesort").drop(columns=["__ro__","__mn__"]).reset_index(drop=True)
    return big

def uso_start_date(preds: pd.DataFrame) -> pd.Timestamp:
    if "date" not in preds.columns:
        raise ValueError("Predictions missing 'date' column.")
    return pd.to_datetime(preds["date"]).min()

def snapshot_row_asof(log: pd.DataFrame, name: str, asof: pd.Timestamp) -> Optional[pd.Series]:
    pk = canon(name)
    sub = log[(log["player_key"]==pk) & (log["date"]<=asof)].sort_values("date")
    if sub.empty: return None
    return sub.iloc[-1]

def compute_field_baseline(log: pd.DataFrame, asof: pd.Timestamp) -> float:
    # baseline upset rate among favorites across field in lookback
    start = asof - pd.Timedelta(days=LOOKBACK_DAYS)
    m = log[(log["date"]>=start) & (log["date"]<=asof)].copy()
    fav = m[m["pwin"] >= FAV_THRESH]
    if fav.empty: return 0.25
    O = int((fav["is_win"]==0).sum())
    N = int(len(fav))
    return O / max(N,1)

def build_usi_table(log: pd.DataFrame, asof: pd.Timestamp, player_names: List[str]) -> pd.DataFrame:
    # precompute per-player frames
    by_player: Dict[str, pd.DataFrame] = {p: log[log["player_key"]==canon(p)].copy() for p in player_names}
    field_q0 = compute_field_baseline(log, asof)

    rows=[]
    for name in player_names:
        dfp = by_player.get(name, pd.DataFrame())
        snap = snapshot_row_asof(log, name, asof)
        if dfp.empty or snap is None:
            rows.append({"name": name, "N_fav":0, "O_upsets":0, "E_exp":0.0, "V_var":0.0,
                         "z":0.0, "z_sigmoid":0.5, "eb_rate":field_q0,
                         "volatility":0.0,"neg_trend":0.0,"fatigue":0.0,"surface_mismatch":0.0,
                         "USI": 100.0*(WEIGHTS["z"]*0.5)})
            continue

        core = compute_core_brittleness(dfp, asof, LOOKBACK_DAYS, field_q0)
        z_sig = sigmoid(core["z"])

        vol = component_volatility(dfp, asof)
        trn = component_trend(dfp, asof)
        fat = component_fatigue(dfp, asof)
        srf = component_surface_mismatch(snap)

        usi_raw = (
            WEIGHTS["z"] * z_sig +
            WEIGHTS["vol"] * vol +
            WEIGHTS["trend"] * trn +
            WEIGHTS["fatigue"] * fat +
            WEIGHTS["surf_mismatch"] * srf
        )
        rows.append({
            "name": name,
            "N_fav": core["N_fav"],
            "O_upsets": core["O_upsets"],
            "E_exp": core["E_exp"],
            "V_var": core["V_var"],
            "z": core["z"],
            "z_sigmoid": z_sig,
            "eb_rate": core["eb_rate"],
            "volatility": vol,
            "neg_trend": trn,
            "fatigue": fat,
            "surface_mismatch": srf,
            "USI": round(100.0 * usi_raw, 1),
        })
    out = pd.DataFrame(rows).sort_values("USI", ascending=False).reset_index(drop=True)
    return out

def augment_predictions_with_usi(preds: pd.DataFrame, usi_df: pd.DataFrame) -> pd.DataFrame:
    usi_map = dict(zip(usi_df["name"].astype(str), usi_df["USI"]))
    def usi(name: str) -> float:
        return float(usi_map.get(alias(str(name)), np.nan))

    P = preds.copy()
    if {"player_a","player_b","prob_player_a_win","prob_player_b_win"}.issubset(P.columns):
        P["favorite_name"] = np.where(P["prob_player_a_win"] >= P["prob_player_b_win"], P["player_a"], P["player_b"])
        P["favorite_prob"] = P[["prob_player_a_win","prob_player_b_win"]].max(axis=1)
        P["dog_name"] = np.where(P["favorite_name"]==P["player_a"], P["player_b"], P["player_a"])
    else:
        # fallback: use confidence and pred_winner if prob columns missing
        P["favorite_name"] = P["pred_winner"]
        P["favorite_prob"] = P.get("confidence", np.nan)
        P["dog_name"] = np.where(P["favorite_name"]==P["player_a"], P["player_b"], P["player_a"])

    P["favorite_usi"] = P["favorite_name"].map(usi)
    P["dog_usi"] = P["dog_name"].map(usi)
    P["usi_gap"] = P["favorite_usi"] - P["dog_usi"]
    # simple flag (tune later)
    P["high_risk_favorite"] = (P["favorite_usi"] >= 70).astype(int)
    return P

# ---------------- Main ----------------
def main() -> None:
    # predictions (for dates & players)
    preds = load_uso_predictions()
    asof = uso_start_date(preds)
    players = sorted(set(alias(x) for x in pd.concat([preds["player_a"], preds["player_b"]]).astype(str)))

    # dataset
    data_path = _first_existing(PREF_DATASETS)
    if not data_path:
        raise FileNotFoundError("No base dataset found. Place a pre-USO snapshot under ./data/ or the raw CSV at project root.")
    matches = pd.read_csv(data_path)
    # strict pre-USO window (no leakage)
    matches["tourney_date"] = matches["tourney_date"].apply(_safe_date)
    pre_uso = matches[matches["tourney_date"] < asof].copy()
    # build player log
    log = build_player_log(pre_uso, EloParams())

    # USI per player (as of tournament start)
    usi = build_usi_table(log, asof, players)
    _ensure_dir(OUT_USI_PLAYERS)
    usi.to_csv(OUT_USI_PLAYERS, index=False, encoding="utf-8-sig")
    print(f"[OK] USI player table → {OUT_USI_PLAYERS}")

    # Augment predictions
    preds_with_usi = augment_predictions_with_usi(preds, usi)
    preds_with_usi.to_csv(OUT_USO_WITH_USI, index=False, encoding="utf-8-sig")
    print(f"[OK] US Open predictions with USI → {OUT_USO_WITH_USI}")

    # preview
    with pd.option_context("display.max_columns", None, "display.width", 180):
        print(usi.head(15).to_string(index=False))
        print(preds_with_usi.head(12).to_string(index=False))

if __name__ == "__main__":
    main()


[OK] USI player table → ./reports/usi\player_usi.csv
[OK] US Open predictions with USI → ./reports/usopen2025_all_rounds_predictions_with_usi.csv
                   name  N_fav  O_upsets     E_exp     V_var         z  z_sigmoid  eb_rate  volatility  neg_trend  fatigue  surface_mismatch  USI
       Denis Shapovalov     15         9  5.313437  3.416441  1.994503   0.880219 0.411552    1.000000   1.000000 0.166667          0.000000 76.3
           Tomas Machac      8         5  2.833124  1.824129  1.604377   0.832629 0.371582    1.000000   1.000000 0.166667          0.001513 74.2
  Felix Auger Aliassime     54        19 15.770412 10.973257  0.974943   0.726104 0.329788    0.973803   1.000000 0.666667          0.000000 73.8
Roberto Carballes Baena      1         1  0.372853  0.233834  1.296927   0.785317 0.304967    0.905504   0.721660 0.166667          0.054139 66.5
            Ethan Quinn      1         1  0.398290  0.239655  1.229118   0.773664 0.304967    0.839406   0.749696 0.166667  